## 1. Install dependencies

In [1]:
!pip install -q segmentation-models-pytorch albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.0 MB/s eta 0:00:00


## 2. Mount Google Drive (recommended)

Saves checkpoints if the Colab session disconnects.

In [2]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/bmw_ai")
DATA_DIR = DRIVE_ROOT / "datasets" / "dmd_yolo"
MODEL_DIR = DRIVE_ROOT / "models"
RUNS_DIR = DRIVE_ROOT / "runs"

for p in (DATA_DIR, MODEL_DIR, RUNS_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("DATA_DIR :", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("RUNS_DIR :", RUNS_DIR)

Mounted at /content/drive
DATA_DIR : /content/drive/MyDrive/bmw_ai/datasets/dmd_yolo
MODEL_DIR: /content/drive/MyDrive/bmw_ai/models
RUNS_DIR : /content/drive/MyDrive/bmw_ai/runs


## 3. Get the dataset into Colab

Pick **one** option below.

### Option A — Upload a ZIP from your PC

Zip your YOLO folder so the archive contains `images/`, `labels/` (and optionally `dataset.yaml`).

In [3]:
# prompt: install and import open dataset as od and download https://www.kaggle.com/datasets/saurabhshahane/mango-varieties-classification

!pip install opendatasets
import opendatasets as od

dataset_url = 'https://www.kaggle.com/datasets/solesensei/solesensei_bdd100k'

od.download(dataset_url)



Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: sayemkabir07
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/solesensei/solesensei_bdd100k


100%|██████████| 7.61G/7.61G [07:14<00:00, 18.8MB/s]


## 4. Imports & Configuration

In [4]:
import os, torch, json
import torch.nn as nn
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import PolynomialLR
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import segmentation_models_pytorch as smp

# Colab Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

IMGSZ = 512
BATCH = 16
EPOCHS = 25

# BDD100K label → our 3-class mapping
BDD_TO_OURS = {
    0:  0,   # road
    8:  0,   # lane marking → road
    10: 0,   # drivable area → road
    1:  1,   # sidewalk → shoulder
    2:  2,   # building → background (default)
}

Using device: cuda


## 5. Dataset Class & Transforms
Python

~2–5 hours on Colab T4 for 100 epochs (reduce `EPOCHS` for a smoke test).

In [5]:
class BDD100KSegDataset(Dataset):
    def __init__(self, root: Path, split: str, transforms=None):
        img_candidates = list(root.rglob("*.jpg"))
        mask_candidates = list(root.rglob("*.png"))

        self.img_dict = {f.stem: f for f in img_candidates if f"/{split}" in str(f).replace("\\", "/").lower() or f"{split}/" in str(f).replace("\\", "/").lower()}

        self.mask_dict = {}
        for f in mask_candidates:
            path_str = str(f).replace("\\", "/").lower()
            if ("label" in path_str and "color" not in path_str) or "mask" in path_str:
                clean_stem = f.stem.replace("_train_id", "").replace("_label", "")
                self.mask_dict[clean_stem] = f

        self.stems = [stem for stem in self.img_dict.keys() if stem in self.mask_dict]
        self.transforms = transforms

        if len(self.stems) == 0:
            raise RuntimeError(f"CRITICAL ERROR: No matching images and masks found for BDD100K '{split}' split! "
                               f"Images found: {len(self.img_dict)}, Masks found: {len(self.mask_dict)}")
        print(f"BDD100K {split}: Found {len(self.stems)} matched image/mask pairs")

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]
        img_path = self.img_dict[stem]

        img = np.array(Image.open(img_path).convert("RGB"))

        # Load grayscale mask and map classes
        raw_mask = np.array(Image.open(self.mask_dict[stem]))
        mask = np.full_like(raw_mask, 2, dtype=np.uint8)  # default background 2
        for bdd_id, our_id in BDD_TO_OURS.items():
            mask[raw_mask == bdd_id] = our_id

        if self.transforms:
            aug = self.transforms(image=img, mask=mask)
            return aug["image"], aug["mask"].long()
        return torch.from_numpy(img).permute(2,0,1).float()/255, torch.from_numpy(mask).long()

train_tfm = A.Compose([
    A.RandomCrop(IMGSZ, IMGSZ),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

val_tfm = A.Compose([
    A.Resize(IMGSZ, IMGSZ),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2(),
])

## DataLoader Setup

In [6]:
# The dataset path matches the extraction path from Cell 2
BDD_ROOT = Path("/content/solesensei_bdd100k/bdd100k_seg")

train_ds = BDD100KSegDataset(BDD_ROOT, "train", train_tfm)
val_ds   = BDD100KSegDataset(BDD_ROOT, "val",   val_tfm)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

BDD100K train: Found 7000 matched image/mask pairs
BDD100K val: Found 1000 matched image/mask pairs
Train: 7000, Val: 1000


## 6. Model, Loss, and Optimizer

In [7]:
model = smp.DeepLabV3Plus(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=3,
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.0, 0.5]).to(DEVICE))
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = PolynomialLR(optimizer, total_iters=EPOCHS, power=0.9)

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

## 7. Training Loop

In [8]:
best_acc = 0.0
MODEL_PATH = "/content/deeplabv3_road.pt"

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for imgs, masks in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    model.eval()
    correct = total_px = 0
    with torch.no_grad():
        for imgs, masks in val_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs).argmax(dim=1)
            correct += (preds == masks).sum().item()
            total_px += masks.numel()

    acc = correct / total_px
    print(f"Epoch {epoch+1:02d} | Loss: {total_loss/len(train_dl):.4f} | Pixel Acc: {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "val_acc": acc}, MODEL_PATH)
        print(f"  ✅ Best model saved (acc={acc:.4f})")

print(f"Done. Best pixel accuracy: {best_acc:.4f}")

Epoch 01 | Loss: 0.3650 | Pixel Acc: 0.8978
  ✅ Best model saved (acc=0.8978)


Epoch 02 | Loss: 0.2786 | Pixel Acc: 0.9120
  ✅ Best model saved (acc=0.9120)


Epoch 03 | Loss: 0.2614 | Pixel Acc: 0.9185
  ✅ Best model saved (acc=0.9185)


Epoch 04 | Loss: 0.2495 | Pixel Acc: 0.9196
  ✅ Best model saved (acc=0.9196)


Epoch 05 | Loss: 0.2364 | Pixel Acc: 0.9198
  ✅ Best model saved (acc=0.9198)


...

Epoch 21 | Loss: 0.1431 | Pixel Acc: 0.9311
  ✅ Best model saved (acc=0.9311)


Epoch 22 | Loss: 0.1385 | Pixel Acc: 0.9295


Epoch 23 | Loss: 0.1351 | Pixel Acc: 0.9324
  ✅ Best model saved (acc=0.9324)


Epoch 24 | Loss: 0.1327 | Pixel Acc: 0.9286


Epoch 25 | Loss: 0.1278 | Pixel Acc: 0.9304
Done. Best pixel accuracy: 0.9324


In [9]:
from google.colab import files

if os.path.exists(MODEL_PATH):
    print("Downloading model...")
    files.download(MODEL_PATH)
else:
    print("Model file not found. Ensure the training loop completed and saved the model.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>